# EIS ML Parameter Extraction
Complete training pipeline from Trials 1-5

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn torch -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

SEED = 42
np.random.seed(SEED)
print('Setup complete')

In [ ]:
def generate_data(n_samples=10000, n_freq=100):
    frequencies = np.logspace(-3, 6, n_freq)
    Rs = np.random.uniform(0.01, 10, n_samples)
    Rct = np.random.uniform(0.1, 100, n_samples)
    Cdl = np.random.uniform(1e-6, 1e-2, n_samples)
    W = np.random.uniform(0.01, 5, n_samples)
    
    spectra = []
    labels = []
    
    for i in range(n_samples):
        f = frequencies
        omega = 2 * np.pi * f
        Z_real = Rs[i] + Rct[i]/(1 + (omega*Cdl[i])**2)
        Z_imag = (omega*Cdl[i]*Rct[i])/(1 + (omega*Cdl[i])**2) + W[i]/np.sqrt(omega + 1e-10)
        spectrum = np.concatenate((Z_real, Z_imag))
        spectra.append(spectrum)
        
        capacitance = Cdl[i]
        energy_density = 0.5 * capacitance * (2.7**2)
        power_density = (2.7**2)/(4*Rct[i])
        labels.append([Rs[i], Rct[i], Cdl[i], W[i], capacitance, energy_density, power_density])
    
    return np.array(spectra), np.array(labels), frequencies

spectra, labels, frequencies = generate_data()
print(f'Dataset: {spectra.shape[0]} samples, {spectra.shape[1]} features')
target_names = ['Rs', 'Rct', 'Cdl', 'Warburg', 'Capacitance', 'Energy Density', 'Power Density']

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(spectra, labels, test_size=0.15, random_state=SEED)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}')

In [ ]:
print('Training Random Forest...')
rf = RandomForestRegressor(n_estimators=300, max_depth=20, random_state=SEED, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_val_scaled)
rf_r2 = r2_score(y_val, rf_preds, multioutput='raw_values')
print(f'RF R2: {rf_r2}')

In [ ]:
print('Training MLP...')
mlp = MLPRegressor(hidden_layer_sizes=(256, 128, 64), max_iter=500, early_stopping=True, random_state=SEED)
mlp.fit(X_train_scaled, y_train)
mlp_preds = mlp.predict(X_val_scaled)
mlp_r2 = r2_score(y_val, mlp_preds, multioutput='raw_values')
print(f'MLP R2: {mlp_r2}')

In [ ]:
import joblib
joblib.dump({'rf': rf, 'mlp': mlp, 'scaler': scaler}, 'models.joblib')
print('Models saved!')
print(f'Best RF: {target_names[np.argmax(rf_r2)]} (R2={rf_r2.max():.4f})')
print(f'Best MLP: {target_names[np.argmax(mlp_r2)]} (R2={mlp_r2.max():.4f})')